<a href="https://colab.research.google.com/github/candeperles/PDI-TP/blob/main/notebooks/04_clasificacion_mobilenetv2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clasificación con MobileNetV2

Tarea 2 de la consigna: clasificar cada imagen en **con paneles / sin paneles** con MobileNetV2 de Keras.

Cada corrida del notebook es **un experimento**, elegido por nombre de `EXPERIMENTOS`. Los definidos cubren
las comparaciones que pide la consigna:

| `EXPERIMENTO` | Qué compara |
|---|---|
| `mobilenetv2_imagenet_congelado` | ImageNet, extractor congelado: solo se entrena la cabeza |
| `mobilenetv2_imagenet_fine_tuning` | ImageNet, cabeza y después la última parte del extractor |
| `mobilenetv2_imagenet_ft_completo` | ImageNet, cabeza y después todo el extractor |
| `mobilenetv2_aleatorio` | Sin pre-entrenamiento: pesos aleatorios, toda la red |

1. Poné tu nombre en `INTEGRANTE` y corré el setup (GPU T4: *Entorno de ejecución → Cambiar tipo de entorno*).
2. Elegí `EXPERIMENTO` y corré todo (*Entorno de ejecución → Ejecutar todas*).
3. Los pesos, las predicciones sobre val y el JSON del experimento quedan en `rutas.resultados` (Drive), con el
   nombre del experimento. Para guardar también los gráficos: *Archivo → Guardar una copia en Drive* con ese nombre.

In [ ]:
# --- Setup: correr siempre primero ---
import os
import subprocess
import sys
from pathlib import Path

INTEGRANTE = os.environ.get("PDI_INTEGRANTE", "nombre")  # reemplazar por tu nombre

if "google.colab" in sys.modules:
    repo = Path("/content/PDI-TP")
    if repo.is_dir():
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/candeperles/PDI-TP.git", str(repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)
else:
    # Funciona al abrir Jupyter desde la raíz o desde notebooks/.
    repo = next((p for p in (Path.cwd(), Path.cwd().parent)
                 if (p / "pdi_tp" / "entorno.py").is_file()), None)
    if repo is None:
        raise FileNotFoundError("Abrí Jupyter desde la raíz del repositorio o desde notebooks/.")
os.chdir(repo)
sys.path.insert(0, str(repo))

from pdi_tp import entorno, datos, resultados, split
rutas = entorno.preparar(INTEGRANTE)


## Configuración del experimento

- `modo="congelado"`: `base.trainable = False`, se entrena solo la cabeza.
- `modo="fine_tuning"`: etapa 1 igual que `congelado`; etapa 2 descongela el extractor desde la capa
  `descongelar_desde` (0 = completo; MobileNetV2 tiene 154 capas) y recompila con un learning rate bajo.
- `modo="completo"`: toda la red entrenable desde el inicio. Es el único modo con sentido para `pesos=None`.

`img_size` es el tamaño con el que entra la imagen a la red (las originales son de 256 px). Keras tiene pesos de
ImageNet para 96, 128, 160, 192 y 224; con otro tamaño avisa y usa los de 224, que también funcionan.

In [ ]:
# Experimentos definidos: cada uno pisa los valores de BASE que cambian. Para una variante nueva,
# agregar una entrada (y hacer push, así queda registrada para el equipo).
EXPERIMENTOS = {
    "mobilenetv2_imagenet_congelado": {"pesos": "imagenet", "modo": "congelado"},
    "mobilenetv2_imagenet_fine_tuning": {"pesos": "imagenet", "modo": "fine_tuning", "descongelar_desde": 100},
    "mobilenetv2_imagenet_ft_completo": {"pesos": "imagenet", "modo": "fine_tuning", "descongelar_desde": 0},
    "mobilenetv2_aleatorio": {"pesos": None, "modo": "completo"},
}

EXPERIMENTO = "mobilenetv2_imagenet_congelado"  # <- elegir uno: nombre del JSON y de los archivos que se guardan

BASE = {
    "modelo": "MobileNetV2",
    "pesos": "imagenet",         # "imagenet" o None (pesos aleatorios)
    "modo": "fine_tuning",       # "congelado", "fine_tuning" o "completo"
    "img_size": 224,
    "batch_size": 32,
    "augmentation": True,
    "dropout": 0.2,
    "class_weight": True,        # compensa el desbalance (hay muchas más imágenes con paneles)
    # Etapa 1 (cabeza, extractor congelado): modos "congelado" y "fine_tuning"
    "epocas_cabeza": 15,
    "lr_cabeza": 1e-3,
    # Etapa 2 (extractor descongelado): modo "fine_tuning"
    "epocas_fine_tuning": 15,
    "lr_fine_tuning": 1e-5,
    "descongelar_desde": 100,    # 0 = todo el extractor
    # Modo "completo": toda la red desde el inicio
    "epocas_completo": 40,
    "lr_completo": 1e-3,
    "paciencia": 6,              # early stopping sobre val_loss
    "umbral": 0.5,
    "semilla": 42,
}

CONFIG = {**BASE, **EXPERIMENTOS[EXPERIMENTO]}
print(EXPERIMENTO, "→", EXPERIMENTOS[EXPERIMENTO])

assert CONFIG["modo"] in ("congelado", "fine_tuning", "completo")
if CONFIG["pesos"] is None and CONFIG["modo"] != "completo":
    raise ValueError("Con pesos aleatorios congelar el extractor no tiene sentido: usar modo='completo'.")

## Datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

keras.utils.set_random_seed(CONFIG["semilla"])
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

archivos = split.cargar_split(rutas)


def cargar(imagenes):
    """Imágenes RGB uint8 (n, 256, 256, 3) y etiquetas 1 = con paneles, 0 = sin paneles."""
    X = np.stack([datos.leer_imagen(p) for p in imagenes])
    y = np.array([datos.tiene_paneles(datos.label_de(rutas, p)) for p in imagenes], dtype="float32")
    return X, y


X_train, y_train = cargar(archivos["train"])
X_val, y_val = cargar(archivos["val"])

pd.DataFrame({
    "imágenes": [len(y_train), len(y_val)],
    "con paneles": [int(y_train.sum()), int(y_val.sum())],
    "sin paneles": [int((1 - y_train).sum()), int((1 - y_val).sum())],
    "% con paneles": [100 * y_train.mean(), 100 * y_val.mean()],
}, index=["train", "val"]).round(1)

El dataset está desbalanceado: la gran mayoría de las imágenes tiene paneles. Un clasificador que siempre diga
"con paneles" ya logra una accuracy alta, así que hay que mirar sobre todo **precision, recall y F1** y la matriz
de confusión. Con `class_weight=True` cada clase pesa en la loss en proporción inversa a su frecuencia.

**Augmentation** (solo en train): rotaciones de 90° y espejados (una foto aérea no tiene "arriba"), y pequeños
cambios de brillo y contraste.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def aumentar(x):
    x = tf.image.rot90(x, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_brightness(x, max_delta=20.0)
    x = tf.image.random_contrast(x, 0.85, 1.15)
    return tf.clip_by_value(x, 0.0, 255.0)


def armar_ds(X, y, entrenamiento):
    ds = tf.data.Dataset.from_tensor_slices((X, y)).map(lambda x, y: (tf.cast(x, tf.float32), y))
    if entrenamiento:
        ds = ds.shuffle(len(X), seed=CONFIG["semilla"])
        if CONFIG["augmentation"]:
            ds = ds.map(lambda x, y: (aumentar(x), y), num_parallel_calls=AUTOTUNE)
    return ds.batch(CONFIG["batch_size"]).prefetch(AUTOTUNE)


train_ds = armar_ds(X_train, y_train, entrenamiento=True)
val_ds = armar_ds(X_val, y_val, entrenamiento=False)  # sin mezclar: el orden coincide con archivos["val"]

if CONFIG["class_weight"]:
    n_pos = y_train.sum()
    n_neg = len(y_train) - n_pos
    class_weight = {0: len(y_train) / (2 * n_neg), 1: len(y_train) / (2 * n_pos)}
else:
    class_weight = None
print("class_weight:", class_weight)

# Un batch aumentado, para chequear que la augmentation tiene sentido
x_ej, y_ej = next(iter(train_ds))
fig, axs = plt.subplots(1, 8, figsize=(16, 2.5))
for ax, x, y in zip(axs, x_ej, y_ej):
    ax.imshow(x.numpy().astype("uint8"))
    ax.set_title("con paneles" if y else "sin paneles", fontsize=9)
    ax.set_axis_off()
plt.show()

## Modelo

`MobileNetV2(include_top=False)` como extractor, y una cabeza propia: *global average pooling → dropout → dense
sigmoide*. El modelo recibe las imágenes en [0, 255] y adentro las redimensiona a `img_size` y las lleva a
[-1, 1], que es lo que espera MobileNetV2 (igual que `mobilenet_v2.preprocess_input`).

Con pesos de ImageNet la base se llama con `training=False`: así sus capas de *batch normalization* siguen usando
las estadísticas de ImageNet aunque después se descongele el extractor. Si no, en el fine-tuning se actualizarían
con batches chicos y se arruinaría lo aprendido (ver la guía de transfer learning de Keras). Con pesos aleatorios
no hay estadísticas que preservar, así que la base se llama normalmente.

In [ ]:
def armar_modelo(config):
    s = config["img_size"]
    base = keras.applications.MobileNetV2(input_shape=(s, s, 3), include_top=False, weights=config["pesos"])

    entradas = keras.Input(shape=(256, 256, 3))
    x = keras.layers.Resizing(s, s)(entradas) if s != 256 else entradas
    x = keras.layers.Rescaling(1 / 127.5, offset=-1)(x)
    x = base(x, training=False) if config["pesos"] == "imagenet" else base(x)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(config["dropout"])(x)
    salidas = keras.layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(entradas, salidas, name=config["modelo"]), base


model, base = armar_modelo(CONFIG)
print(f"Capas del extractor: {len(base.layers)}")
model.summary()

## Entrenamiento

Cada vez que cambia `trainable` hay que volver a compilar: `entrenar` compila y entrena. Se guarda en Drive el
mejor modelo de cada etapa según `val_loss`, y el early stopping restaura esos pesos al terminar.

In [ ]:
historiales = []  # (etapa, history) en orden


def entrenar(etapa, lr, epocas):
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.Precision(name="precision"),
                 keras.metrics.Recall(name="recall"), keras.metrics.AUC(name="auc")],
    )
    entrenables = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
    print(f"--- Etapa {etapa}: lr={lr}, {entrenables:,} parámetros entrenables ---")
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=CONFIG["paciencia"], restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(str(rutas.resultados / f"{EXPERIMENTO}_{etapa}.keras"),
                                        monitor="val_loss", save_best_only=True),
    ]
    hist = model.fit(train_ds, validation_data=val_ds, epochs=epocas,
                     class_weight=class_weight, callbacks=callbacks)
    historiales.append((etapa, hist.history))


if CONFIG["modo"] in ("congelado", "fine_tuning"):
    base.trainable = False
    entrenar("cabeza", CONFIG["lr_cabeza"], CONFIG["epocas_cabeza"])

if CONFIG["modo"] == "fine_tuning":
    base.trainable = True
    for capa in base.layers[:CONFIG["descongelar_desde"]]:
        capa.trainable = False
    entrenar("fine_tuning", CONFIG["lr_fine_tuning"], CONFIG["epocas_fine_tuning"])

if CONFIG["modo"] == "completo":
    base.trainable = True
    entrenar("completo", CONFIG["lr_completo"], CONFIG["epocas_completo"])

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4))
for ax, metrica in zip(axs, ["loss", "accuracy", "recall"]):
    inicio = 0
    for etapa, h in historiales:
        epocas = range(inicio + 1, inicio + len(h[metrica]) + 1)
        ax.plot(epocas, h[metrica], label=f"train ({etapa})")
        ax.plot(epocas, h[f"val_{metrica}"], "--", label=f"val ({etapa})")
        inicio += len(h[metrica])
        if inicio < sum(len(h2[metrica]) for _, h2 in historiales):
            ax.axvline(inicio + 0.5, color="gray", lw=0.8)
    ax.set_title(metrica)
    ax.set_xlabel("época")
axs[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Evaluación sobre val

Las métricas que pide la consigna: matriz de confusión, accuracy, precision, recall y F1, con la clase positiva =
*con paneles*. También se guardan las probabilidades por imagen, para comparar después contra el detector usado
como clasificador sobre las mismas imágenes de val.

In [ ]:
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)

prob_val = model.predict(val_ds, verbose=0).ravel()
pred_val = (prob_val >= CONFIG["umbral"]).astype(int)
y_true = y_val.astype(int)

metricas = {
    "accuracy": accuracy_score(y_true, pred_val),
    "precision": precision_score(y_true, pred_val, zero_division=0),
    "recall": recall_score(y_true, pred_val, zero_division=0),
    "f1": f1_score(y_true, pred_val, zero_division=0),
    "auc": roc_auc_score(y_true, prob_val),
    "accuracy_siempre_con_paneles": float(y_true.mean()),  # referencia: clasificador trivial
    "matriz_confusion": confusion_matrix(y_true, pred_val).tolist(),  # filas = real [sin, con]
    "epocas_por_etapa": {etapa: len(h["loss"]) for etapa, h in historiales},
}
print(classification_report(y_true, pred_val, target_names=["sin paneles", "con paneles"], digits=3))
print({k: round(v, 4) for k, v in metricas.items() if isinstance(v, float)})

ConfusionMatrixDisplay.from_predictions(y_true, pred_val, display_labels=["sin paneles", "con paneles"],
                                        cmap="Blues")
plt.title(EXPERIMENTO)
plt.show()

predicciones = pd.DataFrame({
    "archivo": [p.name for p in archivos["val"]],
    "real": y_true,
    "prob_con_paneles": prob_val,
    "pred": pred_val,
})
predicciones.to_csv(rutas.resultados / f"{EXPERIMENTO}_predicciones_val.csv", index=False)

Errores sobre val: sirven para entender qué le cuesta al modelo (anotarlo para el informe).

In [ ]:
errores = predicciones[predicciones["real"] != predicciones["pred"]]
print(f"{len(errores)} errores de {len(predicciones)}")
n = min(len(errores), 16)
if n:
    fig, axs = plt.subplots(2, 8, figsize=(18, 5))
    for ax in axs.ravel():
        ax.set_axis_off()
    for ax, (i, fila) in zip(axs.ravel(), errores.head(n).iterrows()):
        datos.dibujar_cajas(ax, rutas, archivos["val"][i])
        real = "con" if fila["real"] else "sin"
        ax.set_title(f"real: {real} | p={fila['prob_con_paneles']:.2f}", fontsize=8)
    plt.tight_layout()
    plt.show()

## Guardar resultados

In [ ]:
model.save(rutas.resultados / f"{EXPERIMENTO}.keras")
resultados.guardar_experimento(rutas, nombre=EXPERIMENTO, params=CONFIG, metricas=metricas)

## Comparar con el resto del equipo

In [ ]:
resultados.cargar_experimentos(rutas)